In [ ]:
import torch
from diffusion.approaches.ddpm.ddpm_trainer import DDPMTrainer
from diffusion.sampleables.mnist_sampleable import MNISTSampleable
from diffusion.architectures.backbones.res_unet import ResUnet
from diffusion.approaches.ddpm.backward_process import BackwardProcess
from diffusion.approaches.ddpm.forward_process import ForwardProcess

In [7]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [ ]:
sampeable = MNISTSampleable(train=True)
val_sampleable = MNISTSampleable(train=False)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

forward_process = ForwardProcess(timesteps=10, device=device)
backward_process = BackwardProcess(backbone, forward_process, sampeable.num_classes)

trainer = DDPMTrainer(
    dataset=sampeable,
    val_dataset=val_sampleable,
    forward_process=forward_process,
    backward_process=backward_process,
    backbone=backbone,
    num_classes=sampeable.num_classes,
)

In [4]:
trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

2025-10-11 13:07:24,258 - flow-matching - INFO - Training model with size: 2.734 MiB
Epoch 0/15: 100%|██████████| 500/500 [00:38<00:00, 13.06it/s, loss=0.068485]
2025-10-11 13:08:03,093 - flow-matching - INFO - ['val_loss: 0.033290']
Epoch 1/15: 100%|██████████| 500/500 [00:37<00:00, 13.25it/s, loss=0.032356]
2025-10-11 13:08:41,283 - flow-matching - INFO - ['val_loss: 0.031913']
Epoch 2/15: 100%|██████████| 500/500 [00:37<00:00, 13.25it/s, loss=0.029054]
2025-10-11 13:09:19,476 - flow-matching - INFO - ['val_loss: 0.027996']
Epoch 3/15: 100%|██████████| 500/500 [00:37<00:00, 13.24it/s, loss=0.026890]
2025-10-11 13:09:57,692 - flow-matching - INFO - ['val_loss: 0.025536']
Epoch 4/15: 100%|██████████| 500/500 [00:37<00:00, 13.22it/s, loss=0.025613]
2025-10-11 13:10:35,968 - flow-matching - INFO - ['val_loss: 0.023849']
Epoch 5/15: 100%|██████████| 500/500 [00:37<00:00, 13.17it/s, loss=0.024540]
2025-10-11 13:11:14,383 - flow-matching - INFO - ['val_loss: 0.024333']
Epoch 6/15: 100%|████

In [5]:
torch.save(backbone.state_dict(), "./models/backbone_ddpm.pt")